In [1]:
# we're trying to build an agent that responds to the simple question "what to wear today?" with taking the weather information from an API key.

In [2]:
import os
import certifi
import json
import requests

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

In [3]:
from langchain.agents import create_agent

In [4]:
from rich.console import Console
from rich.markdown import Markdown
from rich.rule import Rule

In [5]:
load_dotenv()

True

In [6]:
def get_weather(city:str) -> dict:
    """returns the weather forecast for a given city"""
    weather_api_key = os.getenv("WEATHER_API_KEY")
    url = "http://api.weatherapi.com/v1/current.json"
    params = {
        "key": weather_api_key,
        "q": city
    }
    r = requests.get(url, params=params)

    if r.status_code == 200: 
        return r.json()
    else:
        return "I was desprete to fetch the weather data!"

In [7]:
weather_info = get_weather("Mashhad")

In [8]:
llm = ChatDeepSeek(model="deepseek-chat", temperature=0.7)

In [9]:
user_personality = {
    "style": ["wearing t-shirt always as first-layer", "in cold days wearing a silk dorse", "barely wears shirts", "in cold days, wearing scarf to keep neck warm"],
    "interests": ["navy blue", "cold environment", "wind", "warm neck", "cold knees", "balanced back", "desprete from the warmness", "hates to sweat!"],
    "geogrophical history": [
        "Living in mashhad with an unbalanced weather during the day, made him always be ready for warm days, knowing that night might be super cold",
        "always carrying jacket during winter & fall to be prepared for the coldness",
        "got used to the diversity of weather"
    ],
    "personal information": [
        "therapist suggested him to wear such to forget the global warmness and risks (his too concerned with global warmness, became desprete to handle daily tasks!)",
        "almost always carrying a backpack and hates to feel warmness or sweat on his back"
    ]
}

In [10]:
prompt = f"Please tell me what to where, knowing {user_personality} and you have access to the weather info tools, as I gave you the access."

In [11]:
response = llm.invoke(prompt)

In [12]:
print(response.content)

I don't actually have access to any weather tools in this conversation — no functions or external tools were provided to me here. So I can't pull live weather for Mashhad right now.

But I can absolutely give you a **wardrobe formula** based on everything you shared. Here's how to dress this person:

---

## Core Principles
1. **T-shirt as base layer, always** — non-negotiable.
2. **Silk dorse (undershirt/vest) added on cold days** — thin, warm, doesn't cause sweat.
3. **No shirts on top** — skip button-ups/overshirts entirely.
4. **Scarf on cold days** — protects the neck, which he hates feeling cold.
5. **Jacket carried (not always worn)** — for Mashhad's day/night temperature swings.
6. **Backpack always** — so back ventilation matters (avoid heavy insulation on the back).

---

## The Formula

### Warm day (daytime Mashhad)
- Cotton or moisture-wicking **T-shirt** (navy blue — his preferred color)
- **No silk dorse**
- Light **jacket in the backpack** (for the evening drop)
- Breat

Now it's time to make this prompt more dynamic and give the LLM access to some internal function that could be ran to find the required information. We're trying to enhance this prompt and the pipeline<br>
Here's the previous prompt: <br> 
prompt = f"Please tell me what to where, knowing {user_personality} and weather info is {weather_info}"

In [13]:
tools = [get_weather]

In [14]:
agent_executor = create_agent(
    model="deepseek-chat",
    tools = tools,
    system_prompt=prompt,
)

In [15]:
result = agent_executor.invoke({"messages": [{"role": "user", "content": "What should I wear today in Mashhad?"}]})

In [16]:
console = Console()

In [17]:
console.print(result["messages"][3].content)

Here's your Mashhad forecast for today (Oct 10, 10:14 local):

- **Now:** 22.8°C, sunny, feels like 18.8°C
- **Wind:** 14.8 km/h ESE (gusts up to 25.6 km/h)
- **Humidity:** 31% (dry), no rain expected
- **UV:** 5.3 (moderate — worth noting)

## What to wear

Given your profile (always t-shirt as base layer, silk dorse in cold, scarf for the neck, backpack, hates sweat, 
needs a **balanced back**, prepared for temperature swings):

**Daytime (now):**
- **T-shirt** as your first layer — it's mild and you'll likely be out in the sun.
- **Light jacket** carried in/on your backpack — you already do this in fall/winter, and with a 4°C "feels like" 
drop and gusts up to 25 km/h, the wind will make it feel cooler than the 23°C suggests. This also keeps your back 
balanced rather than bundled.
- **No heavy layer needed** — humidity is low and it's dry, so you won't overheat or sweat on your back.

**Evening (your key concern):**
- Mashhad nights drop hard. Bring your **scarf** for the neck — this is exactly the "warm neck, cool core" balance 
you prefer.
- **Silk dorse** on standby if it turns genuinely cold after sunset.

**Notes for you specifically:**
- The dry, windy air means evaporation will keep you from sweating on your back — good for the backpack situation.
- UV is moderate (5.3); a light cap or sunglasses wouldn't hurt if you're out long.
- Your therapist's advice holds well today: dress for *this* day's real numbers, not for global warming worry — 
it's a comfortable, balanced, layer-friendly Mashhad day. 🌤️